# Code mẫu — Phân tích dữ liệu trước huấn luyện

Khám phá dữ liệu, hay EDA, chỉ có ích khi một điều ta phát hiện ra làm thay đổi cách đọc dữ liệu, cách chia, hay chọn đặc trưng. Ba tình huống nhỏ dưới đây lần lượt xét dữ liệu thiếu, nhiều mẫu chung một nguồn, và thời điểm dự đoán. Tất cả đều nên làm trước khi nhìn vào điểm của model.

## Ví dụ 1: Được dùng median nào để điền chỗ thiếu?

Bốn số đo train là `2, NaN, 6, 10`. Validation có `100, NaN`. Tính tay, median của train là 6. Nếu gộp cả validation vào để học median, giá trị 100 sẽ kéo median đi, dù ta chưa hề dùng tới nhãn của validation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

train = pd.DataFrame({"id": ["001", "002", "002", "004"],
                      "signal": [2.0, np.nan, 6.0, 10.0]})
validation = np.array([[100.0], [np.nan]])
print("thiếu:", int(train["signal"].isna().sum()))
print("ID lặp:", train.loc[train["id"].duplicated(), "id"].tolist())
X_train = train[["signal"]].to_numpy()
clean = SimpleImputer(strategy="median").fit(X_train)
leaky = SimpleImputer(strategy="median").fit(np.vstack([X_train, validation]))
assert clean.statistics_[0] == 6.0 and leaky.statistics_[0] == 8.0
print("median train:", clean.statistics_.tolist())
print("median ghép cả validation:", leaky.statistics_.tolist())
print("validation sau transform sạch:", clean.transform(validation).ravel().tolist())

thiếu: 1
ID lặp: ['002']
median train: [6.0]
median ghép cả validation: [8.0]
validation sau transform sạch: [100.0, 6.0]


Bỏ ô thiếu, train còn `2, 6, 10`, median là 6. Gộp thêm 100 thì thành `2, 6, 10, 100`, median là trung bình của hai số giữa, tức 8. Validation đã len vào phép biến đổi. Cách làm đúng là học median từ train, rồi dùng nó để điền cho validation, nên ô thiếu của validation được điền bằng 6.

Bảng còn có ID `002` bị lặp. Đó là dấu hiệu cần kiểm lại nguồn dữ liệu. Đừng xóa ngay, vì hai dòng có số đo khác nhau, và ta chưa biết dòng nào đúng. Bộ điền chỉ lo chỗ thiếu, chứ không lo chuyện ID.

## Ví dụ 2: Nhớ một tài liệu khác với đoán đúng tài liệu mới

Có bốn tài liệu A tới D, mỗi tài liệu có hai đoạn. Đặc trưng của mỗi đoạn là một vector one-hot, cho biết đoạn đó thuộc tài liệu nào. Nhãn của bốn tài liệu là `0, 1, 1, 0`. Không có quy luật nào để đoán nhãn của một tài liệu mới từ đặc trưng này. Ta chia theo hai cách: chia từng đoạn, mỗi tài liệu một đoạn sang mỗi phía, và chia theo cả tài liệu bằng `GroupShuffleSplit`.

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

groups = np.repeat(np.arange(4), 2)
X = np.eye(4)[groups]
y = np.array([0, 1, 1, 0])[groups]
row_train, row_val = np.array([0, 2, 4, 6]), np.array([1, 3, 5, 7])
splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0)
group_train, group_val = next(splitter.split(X, y, groups))
for name, fit_rows, val_rows in [
    ("theo đoạn", row_train, row_val), ("theo tài liệu", group_train, group_val)
]:
    overlap = sorted(map(int, set(groups[fit_rows]) & set(groups[val_rows])))
    model = KNeighborsClassifier(n_neighbors=1).fit(X[fit_rows], y[fit_rows])
    prediction = model.predict(X[val_rows])
    print(name, "nguồn chung:", overlap,
          "accuracy:", f"{accuracy_score(y[val_rows], prediction):.3f}")
    print("nhãn thật/dự đoán:", y[val_rows].tolist(), prediction.tolist())
assert not (set(groups[group_train]) & set(groups[group_val]))
assert np.array_equal(X[row_train], X[row_val])

theo đoạn nguồn chung: [0, 1, 2, 3] accuracy: 1.000
nhãn thật/dự đoán: [0, 1, 1, 0] [0, 1, 1, 0]
theo tài liệu nguồn chung: [] accuracy: 0.500
nhãn thật/dự đoán: [1, 1, 0, 0] [0, 0, 0, 0]


Chia theo đoạn, cả bốn tài liệu đều có mặt ở hai phía, và đặc trưng train giống hệt đặc trưng validation. Model láng giềng gần nhất chỉ việc nhớ tài liệu nào mang nhãn nào, và đạt accuracy 1. Con số này đo khả năng nhớ, chứ không đo khả năng đoán tài liệu mới.

Chia theo tài liệu thì không có tài liệu chung nào. Vector của tài liệu mới cách đều mọi vector train, nên model chỉ còn biết đoán theo luật phá hòa, và accuracy rơi xuống 0.5. Con số 0.5 này phụ thuộc vào việc tài liệu nào rơi vào validation và cách phá hòa. Điều chắc chắn là: với đặc trưng này, model không học được gì dùng cho tài liệu mới.

## Ví dụ 3: Chia theo thời gian, và một cảm biến bị lệch

Phần đầu dùng mười hai thời điểm cách đều nhau để kiểm các fold. Mỗi fold phải có train nằm hoàn toàn trước validation, và chừa một thời điểm ở giữa làm khoảng đệm. Phần sau xét một cảm biến mới, đọc cao hơn cảm biến cũ đúng 10 đơn vị. Độ lệch này là thông số hiệu chuẩn đã biết trước khi dự đoán. Nó không được ước lượng từ nhãn của tương lai.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

times = np.arange(12)
for number, (fit_rows, val_rows) in enumerate(
    TimeSeriesSplit(n_splits=3, test_size=2, gap=1).split(times), 1
):
    assert times[fit_rows].max() + 1 < times[val_rows].min()
    print("fold", number, "train cuối:", int(times[fit_rows].max()),
          "val đầu:", int(times[val_rows].min()))

X_train = np.array([0.0, 1.0, 3.0, 4.0]).reshape(-1, 1)
y_train = np.array([0, 0, 1, 1])
future = pd.DataFrame({"device": ["old", "old", "new", "new"],
                       "reading": [1.0, 3.0, 11.0, 13.0],
                       "known_offset": [0.0, 0.0, 10.0, 10.0],
                       "label": [0, 1, 0, 1]})
model = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_train)
raw = model.predict(future[["reading"]].to_numpy())
corrected = model.predict((future["reading"] - future["known_offset"]).to_numpy().reshape(-1, 1))
for device in ["old", "new"]:
    rows = future["device"].eq(device).to_numpy()
    labels = future.loc[rows, "label"].to_numpy()
    print(device, "n:", int(rows.sum()), "raw/corrected:",
          f"{accuracy_score(labels, raw[rows]):.2f}",
          f"{accuracy_score(labels, corrected[rows]):.2f}")
assert np.array_equal(corrected, future["label"].to_numpy())

fold 1 train cuối: 4 val đầu: 6
fold 2 train cuối: 6 val đầu: 8
fold 3 train cuối: 8 val đầu: 10
old n: 2 raw/corrected: 1.00 1.00
new n: 2 raw/corrected: 0.50 1.00


Ở fold đầu, train kết thúc ở thời điểm 4, thời điểm 5 bị bỏ làm khoảng đệm, và validation bắt đầu ở 6. Khoảng đệm giúp tránh trường hợp mẫu cuối của train và mẫu đầu của validation quá gần nhau, gần như mang cùng thông tin.

Cây quyết định học được một ngưỡng nằm giữa 1 và 3. Với cảm biến cũ, nó đúng cả hai mẫu. Với cảm biến mới, cả hai số đọc thô, 11 và 13, đều vượt ngưỡng, nên một trong hai mẫu bị đoán sai. Trừ đi độ lệch đã biết, số đọc trở về cùng thang đo như lúc train, và cả hai mẫu đều đúng.

Ở đây độ lệch được tạo ra có chủ đích, nên ta biết rõ nguyên nhân. Với dữ liệu thật, thấy hai biểu đồ phân bố khác nhau thôi thì chưa đủ để biết nguyên nhân. Và muốn hiệu chỉnh, thông số hiệu chỉnh phải có sẵn ngay lúc dự đoán.